# 03 - Few-Shot Prompting (RQ2)

Does supplying labeled PropBank demonstrations improve SRL?
Runs 2-, 5-, and 10-shot configurations with demonstrations drawn from the
**train** split (never from test).

**Deliverable 3:** few-shot predictions under each configuration.

In [ ]:
import sys, os
from pathlib import Path
# Make src/ importable whether run from repo root or notebooks/.
ROOT = Path.cwd()
if (ROOT / 'src').exists():
    sys.path.insert(0, str(ROOT / 'src'))
elif (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)
print('project root:', ROOT)


In [ ]:
import data_loader as dl, srl_inference as si, evaluation as ev
import prompts as pr
client = si.get_client()
MODEL = si.DEFAULT_MODEL
train = dl.read_jsonl('data/train.jsonl')
test = dl.read_jsonl('data/test.jsonl')

## Select demonstrations (role-coverage biased, fixed across test sentences)

In [ ]:
demo_sets = {k: pr.select_demos(train, k, seed=7) for k in (2, 5, 10)}
for k, d in demo_sets.items():
    print(f'{k}-shot demos:', [e.predicate for e in d])

## Run each configuration

In [ ]:
summaries = []
for k, demos in demo_sets.items():
    res = si.predict_batch(client, test, model=MODEL, demos=demos)
    si.save_predictions(res, f'results/predictions_few_shot_{k}.jsonl')
    m = ev.evaluate(res, test)
    summaries.append(ev.summary_row(f'{k}-shot', m))
import pandas as pd
pd.DataFrame(summaries)